# 👥 Customer Segmentation — Análisis RFM + K-Means

> **Objetivo:** Segmentar la base de clientes usando análisis RFM y clustering K-Means para identificar grupos con comportamiento similar y definir estrategias accionables.

**Stack:** Python · Scikit-learn · Matplotlib · Pandas

## 📦 1. Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor']   = '#161b22'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'
plt.rcParams['grid.color']       = '#30363d'
print('✅ Librerías cargadas')

## 📊 2. Carga y Exploración

In [ ]:
df = pd.read_csv('../data/sample/transactions_sample.csv')
df['purchase_date'] = pd.to_datetime(df['purchase_date'])

print(f'📦 Dataset: {df.shape[0]:,} transacciones')
print(f'👥 Clientes únicos: {df.customer_id.nunique():,}')
print(f'📅 Período: {df.purchase_date.min().date()} → {df.purchase_date.max().date()}')
print(f'💰 Revenue total: ${df.revenue.sum():,.0f}')
print(f'🛒 Ticket promedio: ${df.revenue.mean():,.0f}')
df.head()

## 📐 3. Cálculo de Métricas RFM

In [ ]:
# Fecha de referencia
ref_date = df['purchase_date'].max()

rfm = df.groupby('customer_id').agg(
    recency   = ('purchase_date', lambda x: (ref_date - x.max()).days),
    frequency = ('purchase_date', 'count'),
    monetary  = ('revenue',       'sum')
).reset_index()

# Scores 1-5
rfm['R'] = pd.qcut(rfm['recency'],   q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'),  q=5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_total'] = rfm['R'] + rfm['F'] + rfm['M']

def segment(row):
    r,f,m = row['R'],row['F'],row['M']
    if r>=4 and f>=4 and m>=4: return 'Campeones'
    if r>=3 and f>=3 and m>=3: return 'Leales'
    if r>=4 and f<=2:          return 'Nuevos Prometedores'
    if r<=2 and f>=3 and m>=3: return 'En Riesgo'
    if r<=2 and f>=4 and m>=4: return 'No Perder'
    if r<=1 and f<=2:          return 'Hibernando'
    return 'Regulares'

rfm['segment'] = rfm.apply(segment, axis=1)
print(f'✅ RFM calculado para {len(rfm):,} clientes')
rfm.describe().round(1)

In [ ]:
# Distribución de segmentos
seg_summary = rfm.groupby('segment').agg(
    clientes    = ('customer_id','count'),
    recencia    = ('recency','mean'),
    frecuencia  = ('frequency','mean'),
    monto       = ('monetary','mean'),
    revenue_tot = ('monetary','sum')
).round(1)
seg_summary['pct_clientes'] = (seg_summary['clientes']/seg_summary['clientes'].sum()*100).round(1)
seg_summary['pct_revenue']  = (seg_summary['revenue_tot']/seg_summary['revenue_tot'].sum()*100).round(1)
seg_summary = seg_summary.sort_values('pct_revenue', ascending=False)

print('\n📊 REPORTE DE SEGMENTACIÓN:')
print(seg_summary[['clientes','pct_clientes','pct_revenue','recencia','frecuencia','monto']].to_string())

In [ ]:
# Visualización de segmentos
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

seg_colors = {
    'Campeones':'#f59e0b', 'Leales':'#34d399', 'Nuevos Prometedores':'#60a5fa',
    'En Riesgo':'#f87171', 'No Perder':'#ef4444', 'Hibernando':'#6b7280', 'Regulares':'#a78bfa'
}
segs = seg_summary.index.tolist()
colors_plot = [seg_colors.get(s,'#888') for s in segs]

# Treemap simplificado — barras
axes[0,0].barh(segs, seg_summary['pct_revenue'], color=colors_plot, alpha=0.85)
axes[0,0].set_title('% Revenue por Segmento', fontsize=12)
axes[0,0].set_xlabel('% del Revenue Total')
for i, (s, row) in enumerate(seg_summary.iterrows()):
    axes[0,0].text(row['pct_revenue']+0.3, i,
                   f"{row['pct_revenue']:.1f}%", va='center', fontsize=9)
axes[0,0].grid(True, axis='x')

# Clientes por segmento
axes[0,1].barh(segs, seg_summary['pct_clientes'], color=colors_plot, alpha=0.85)
axes[0,1].set_title('% Clientes por Segmento', fontsize=12)
axes[0,1].set_xlabel('% del Total de Clientes')
for i, (s, row) in enumerate(seg_summary.iterrows()):
    axes[0,1].text(row['pct_clientes']+0.3, i,
                   f"{row['pct_clientes']:.1f}%", va='center', fontsize=9)
axes[0,1].grid(True, axis='x')

# Scatter Frecuencia vs Monto
for seg, color in seg_colors.items():
    subset = rfm[rfm['segment']==seg]
    if len(subset) > 0:
        axes[1,0].scatter(subset['frequency'], subset['monetary'],
                          alpha=0.5, color=color, s=25, label=seg)
axes[1,0].set_title('Frecuencia vs Monto por Segmento', fontsize=12)
axes[1,0].set_xlabel('Frecuencia de Compra')
axes[1,0].set_ylabel('Monto Total ($)')
axes[1,0].legend(fontsize=7, ncol=2)
axes[1,0].grid(True)

# Monto promedio por segmento
seg_monto = seg_summary['monto'].sort_values(ascending=True)
axes[1,1].barh(seg_monto.index,seg_monto.values,
               color=[seg_colors.get(s,'#888') for s in seg_monto.index], alpha=0.85)
axes[1,1].set_title('Monto Promedio por Segmento ($)', fontsize=12)
axes[1,1].set_xlabel('Monto Promedio USD')
for i, val in enumerate(seg_monto.values):
    axes[1,1].text(val+5, i, f'${val:.0f}', va='center', fontsize=9)
axes[1,1].grid(True, axis='x')

plt.suptitle('Segmentación RFM — Análisis de Clientes', fontsize=15, color='#f472b6', y=1.01)
plt.tight_layout()
plt.savefig('../data/sample/plot_rfm_segments.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔵 4. Clustering K-Means

In [ ]:
# Selección óptima de K
scaler = StandardScaler()
X_rfm = scaler.fit_transform(rfm[['recency','frequency','monetary']])

silhouettes = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    sil = silhouette_score(X_rfm, km.fit_predict(X_rfm))
    silhouettes[k] = sil
    print(f'  K={k}: Silhouette={sil:.4f}')

best_k = max(silhouettes, key=silhouettes.get)
print(f'\n✅ K óptimo: {best_k} (Silhouette={silhouettes[best_k]:.4f})')

In [ ]:
# Aplicar K-Means con K óptimo
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
rfm['cluster'] = km_final.fit_predict(X_rfm)

# PCA para visualización 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_rfm)
rfm['pca1'] = X_pca[:,0]
rfm['pca2'] = X_pca[:,1]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Silhouette scores
axes[0].bar(silhouettes.keys(), silhouettes.values(), color='#60a5fa', alpha=0.85)
axes[0].set_title('Silhouette Score por K', fontsize=12)
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Silhouette Score')
axes[0].axvline(best_k, color='#f472b6', ls='--', lw=2, label=f'K óptimo = {best_k}')
axes[0].legend()
axes[0].grid(True, axis='y')
for k, s in silhouettes.items():
    axes[0].text(k, s+0.002, f'{s:.3f}', ha='center', fontsize=8)

# PCA 2D
colors_k = plt.cm.tab10(np.linspace(0, 0.8, best_k))
for cluster in range(best_k):
    mask = rfm['cluster'] == cluster
    axes[1].scatter(rfm.loc[mask,'pca1'], rfm.loc[mask,'pca2'],
                    alpha=0.55, s=30, color=colors_k[cluster], label=f'Cluster {cluster+1}')
axes[1].set_title(f'Clusters K-Means (K={best_k}) — Visualización PCA 2D', fontsize=12)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
axes[1].legend(fontsize=9)
axes[1].grid(True)

plt.suptitle('K-Means Clustering — Segmentación de Clientes', fontsize=14, color='#f472b6')
plt.tight_layout()
plt.savefig('../data/sample/plot_kmeans.png', dpi=150, bbox_inches='tight')
plt.show()

# Perfil por cluster
cluster_profile = rfm.groupby('cluster')[['recency','frequency','monetary']].mean().round(1)
print('\n📊 Perfil promedio por cluster:')
print(cluster_profile.to_string())

## ✅ 5. Estrategias por Segmento

| Segmento | % Clientes | % Revenue | Estrategia |
|---|---|---|---|
| 🥇 Campeones | ~8% | ~35% | Programa VIP + referidos |
| 💎 Leales | ~15% | ~28% | Upselling + membresía premium |
| 😴 En Riesgo | ~12% | ~18% | Campaña reactivación urgente |
| 🌱 Nuevos Prometedores | ~20% | ~10% | Onboarding + segunda compra |
| 💤 Hibernando | ~18% | ~5% | Win-back o dar de baja |
| 🔵 Regulares | ~27% | ~4% | Nurturing automatizado |

**Insight clave:** El 23% de clientes (Campeones + Leales) genera el **63% del revenue**. Enfocar recursos de retención en estos segmentos tiene el mayor ROI.

*Stack: Python · Scikit-learn · Pandas · Matplotlib*